# Bellabeat Case Study — Turning Sleep into the Gateway to Retention
## Analysis notebook (final, verified)

End-to-end analysis for the Google Data Analytics capstone: audit → clean → classify → analyze → visualize. Every number in the written deliverables (README, REPORT, ANALYZE, CLEANING_LOG, PREPARE, ASK) traces to a cell of this notebook.

### Method
Google Data Analytics cycle: Ask → Prepare → Process → Analyze → Share. Python (pandas, matplotlib) on the Kaggle dataset `arashnic/fitbit` (FitBit Fitness Tracker Data, Möbius, CC0). Analysis window: the 04.12.16-05.12.16 export, 2016-04-12 to 2016-05-12.

### Verified dataset facts (re-checked by Section 2 of this notebook)
- 33 unique users in dailyActivity (the Kaggle description announces 30 — documented discrepancy)
- 24 sleep users, 14 heart-rate users, 8 weight users
- 3 exact duplicates in sleepDay (413 → 410 rows), 940 activity rows, 31 distinct days

### Sleep-adoption classification — rules stated in code
Ordered, exhaustive, auditable: every one of the 33 users falls in exactly one class.

| Class | Rule (in execution order) | Result |
|---|---|---|
| never | zero recorded nights | 9 users (27%) |
| regular | at least 25 of the 31 study nights tracked | 10 users (30%) |
| abandoned | last recorded night at least 14 days before study end | 3 users (9%) |
| irregular | every remaining night-wearer (2-24 nights) | 11 users (33%) |

Sum: 33 of 33 users. The full per-user table is printed by Section 3 and exported to `user_classification.csv`.

### Two-layer discipline
Findings below are FACTS from this 33-user 2016 sample. Market claims (sleep market size, smart-ring growth, perimenopause statistics) are sourced separately in SOURCES.md and framed as hypotheses to validate — never as findings. The dataset cannot describe women or today's market; it identifies durable behavioral structures only.

### How to run
Kaggle → Add Input → `arashnic/fitbit` → Run All. The data path is auto-detected; the correct export folder is selected automatically.

In [ ]:
# ---------------------------------------------------------------------------
# 0. SETUP — imports, plotting style, data-path resolution
# ---------------------------------------------------------------------------
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

print("Environment: Python", sys.version.split()[0],
      "| pandas", pd.__version__, "| numpy", np.__version__,
      "| matplotlib", mpl.__version__)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "font.size": 10,
})

# The Kaggle archive contains TWO export windows (03.12.16-04.11.16 and
# 04.12.16-05.12.16); this analysis uses the SECOND one.
CANDIDATE_DIRS = [
    Path("/kaggle/input/datasets/arashnic/fitbit"),
    Path("/kaggle/input/arashnic/fitbit"),
    Path("/kaggle/input/fitbit"),
    Path("/kaggle/input"),
    Path.cwd(),
]

def find_data_dir(anchor="dailyActivity_merged.csv"):
    """Return the directory of the 04.12.16-05.12.16 export (analysis window)."""
    hits = []
    for d in CANDIDATE_DIRS:
        try:
            if d.exists():
                hits.extend(sorted(d.rglob(anchor)))
        except OSError:
            pass
    if not hits:
        diag = []
        for d in CANDIDATE_DIRS[:-1]:
            try:
                if d.exists():
                    inside = ", ".join(sorted(p.name for p in d.iterdir()))
                    diag.append(f"  {d} -> {inside or 'empty'}")
                else:
                    diag.append(f"  {d} -> not mounted")
            except OSError as e:
                diag.append(f"  {d} -> unreadable ({e})")
        raise FileNotFoundError(
            f"{anchor} was not found.\n"
            "Fix on Kaggle: right panel -> 'Add Input' -> search "
            "'FitBit Fitness Tracker Data' (arashnic/fitbit) -> attach -> re-run.\n"
            "Searched:\n" + "\n".join(diag)
        )
    preferred = [h for h in hits if "4.12.16" in str(h.parent)]
    chosen = (preferred or hits)[0]
    print("dailyActivity_merged.csv found in:")
    for h in hits:
        print("  -", h.parent, " <-- analysis export" if h == chosen else "")
    return chosen.parent

DATA_DIR = find_data_dir()
print("\nData directory :", DATA_DIR)
files = sorted(DATA_DIR.glob("*.csv"))
print(f"{len(files)} CSV files in the analysis export:")
for f in files:
    print("  -", f.name)

## 1. Load the analysis files

Three files are analysed (daily activity, daily sleep, hourly intensity); two are loaded only for the audit (heart rate, weight log). Dates use the US format of the Kaggle export (MM/DD/YYYY, 12-hour clock).

In [ ]:
# ---------------------------------------------------------------------------
# 1. LOAD — tolerant date parsing
# ---------------------------------------------------------------------------
DT = "%m/%d/%Y %I:%M:%S %p"     # Kaggle export timestamp format (US, 12h clock)

def parse_dt(series, fmt):
    """Parse with the documented export format; infer only if that fails."""
    parsed = pd.to_datetime(series, format=fmt, errors="coerce")
    if parsed.isna().all():
        parsed = pd.to_datetime(series, errors="coerce")
    return parsed

daily  = pd.read_csv(DATA_DIR / "dailyActivity_merged.csv")
sleep  = pd.read_csv(DATA_DIR / "sleepDay_merged.csv")
hourly = pd.read_csv(DATA_DIR / "hourlyIntensities_merged.csv")
hr     = pd.read_csv(DATA_DIR / "heartrate_seconds_merged.csv")
weight = pd.read_csv(DATA_DIR / "weightLogInfo_merged.csv")

daily["ActivityDate"]  = parse_dt(daily["ActivityDate"], "%m/%d/%Y")
sleep["SleepDay"]      = parse_dt(sleep["SleepDay"], DT)
hourly["ActivityHour"] = parse_dt(hourly["ActivityHour"], DT)

assert daily["ActivityDate"].notna().any(), "ActivityDate did not parse"
assert sleep["SleepDay"].notna().any(), "SleepDay did not parse"

print("Loaded:")
for name, df in [("dailyActivity", daily), ("sleepDay", sleep),
                 ("hourlyIntensities", hourly), ("heartrate_seconds", hr),
                 ("weightLogInfo", weight)]:
    print(f"  {name:<20} {df.shape[0]:>9,} rows x {df.shape[1]:>2} cols")

## 2. Audit — the raw files, verified before any analysis

Expected values come from the case study documentation; each is compared to what the data actually contains. The Kaggle description announces 30 users; the verified count is 33 — a documented data-quality finding.

In [ ]:
# ---------------------------------------------------------------------------
# 2. AUDIT — verify the raw files
# ---------------------------------------------------------------------------
START  = daily["ActivityDate"].min().normalize()
END    = daily["ActivityDate"].max().normalize()
N_DAYS = (END - START).days + 1

def audit(label, computed, expected):
    verdict = "CONFIRMED" if computed == expected else f"DIFFERS (expected {expected})"
    print(f"{label:<52} {computed!s:<18} {verdict}")
    return computed

print("=== Unique users (Id) ===")
n_act = audit("dailyActivity users (Kaggle announces 30)", daily["Id"].nunique(), 33)
n_slp = audit("sleepDay users",   sleep["Id"].nunique(), 24)
n_hr  = audit("heartrate users",  hr["Id"].nunique(), 14)
n_wgt = audit("weightLog users", weight["Id"].nunique(), 8)

# cross-file integrity: every sleep user must exist in the activity file
orphan_sleep_users = set(sleep["Id"]) - set(daily["Id"])
print(f"{'sleepDay users absent from dailyActivity':<52} {len(orphan_sleep_users)!s:<18} {'expected 0'}")

print("\n=== Duplicates ===")
dup_sleep = int(sleep.duplicated().sum())
audit("sleepDay exact-duplicate rows", dup_sleep, 3)
sleep = sleep.drop_duplicates().reset_index(drop=True)
audit("sleepDay rows after de-duplication", len(sleep), 410)
audit("dailyActivity exact-duplicate rows", int(daily.duplicated().sum()), 0)

print("\n=== Date coverage ===")
print("Study window :", START.date(), "->", END.date())
audit("distinct activity days", daily["ActivityDate"].nunique(), 31)
audit("dailyActivity rows", len(daily), 940)
print("Rows per active user (mean):", round(len(daily) / daily["Id"].nunique(), 1),
      " -> engagement is irregular; missing days are kept as a behavioural fact")

## 3. Sleep-adoption classification — explicit, ordered, exhaustive

Rules (stated in code, therefore reproducible and auditable):
1. **never** — zero recorded nights;
2. **regular** — at least 25 of the 31 study nights tracked;
3. **abandoned** — last recorded night at least 14 days before the study end (tracking stopped early);
4. **irregular** — every remaining night-wearer (intermittent across the month).

The four classes sum to exactly 33. The full per-user table is printed so any reviewer can audit the assignment line by line.

In [ ]:
# ---------------------------------------------------------------------------
# 3. PER-USER TABLE + EXHAUSTIVE SLEEP CLASSIFICATION
# ---------------------------------------------------------------------------
u = (daily.groupby("Id")
        .agg(days_active=("ActivityDate", "nunique"),
             mean_steps=("TotalSteps", "mean"),
             total_steps=("TotalSteps", "sum"))
        .reset_index())

s = (sleep.groupby("Id")
         .agg(nights=("SleepDay", "nunique"),
              first_night=("SleepDay", "min"),
              last_night=("SleepDay", "max"))
         .reset_index())

u = u.merge(s, on="Id", how="left")
u["nights"] = u["nights"].fillna(0).astype(int)

REGULAR_MIN_NIGHTS = 25   # "regular"   = tracked >= 25 of the 31 study nights
STOPPED_EARLY_DAYS = 14   # "abandoned" = last night >= 14 days before study end

def classify_sleep(row):
    """Ordered, exhaustive: every one of the 33 users gets exactly one class."""
    if row["nights"] == 0:
        return "never"
    if row["nights"] >= REGULAR_MIN_NIGHTS:
        return "regular"
    if (END - row["last_night"]).days >= STOPPED_EARLY_DAYS:
        return "abandoned"
    return "irregular"

u["sleep_class"] = u.apply(classify_sleep, axis=1)

assert len(u) == 33, "the per-user table must cover exactly 33 users"
assert u["sleep_class"].isin(["never", "abandoned", "irregular", "regular"]).all()
assert u["sleep_class"].value_counts().sum() == 33, "classes must sum to 33"

counts = u["sleep_class"].value_counts().reindex(
    ["never", "abandoned", "irregular", "regular"]).fillna(0).astype(int)

print("=== Sleep adoption among the 33 daily-active users ===")
for k in ["never", "abandoned", "irregular", "regular"]:
    v = int(counts[k])
    print(f"  {k:<10} {v:>2} users ({v / 33 * 100:.0f}%)")
print(f"  sum: {int(counts.sum())} of 33 users")
print(f"Users who ever recorded a night: {n_slp} of 33 ({n_slp / 33 * 100:.0f}%); "
      f"durable adoption: {int(counts['regular'])} of 33 ({counts['regular'] / 33 * 100:.0f}%)")

print("\n=== Full per-user classification (auditable line by line) ===")
u_view = u.sort_values(["sleep_class", "nights"]).copy()
u_view["pct_nights"] = (u_view["nights"] / N_DAYS * 100).round(0).astype(int)
print(u_view[["Id", "days_active", "nights", "pct_nights",
              "first_night", "last_night", "sleep_class"]].to_string(index=False))

u.to_csv("user_classification.csv", index=False)
print("\nSaved: user_classification.csv")

## 4. Activity trajectory and the four personas

Segments map onto the sleep classes: **Completes** (regular), **Inconsistent** (irregular), **Engaged day-only** (never), **Decayers** (abandoned).

In [ ]:
# ---------------------------------------------------------------------------
# 4. WEEKLY TRAJECTORY + FOUR PERSONAS (exhaustive by construction)
# ---------------------------------------------------------------------------
daily["week"] = ((daily["ActivityDate"] - START).dt.days // 7) + 1   # weeks 1-5, week 5 partial (3 days)

weekly = daily.groupby(["Id", "week"])["TotalSteps"].mean().unstack()

# trajectory = late full week vs week 1 ("three weeks later")
W_EARLY, W_LATE = 1, 4
if W_EARLY not in weekly.columns or W_LATE not in weekly.columns:
    available = [w for w in weekly.columns if weekly[w].notna().any()]
    W_EARLY = available[0]
    W_LATE = available[-1]
    print(f"NOTE: weeks 1/4 not both present -> using weeks {W_EARLY} and {W_LATE}.")
traj = (weekly[W_LATE] / weekly[W_EARLY] - 1).rename("trajectory")
u = u.merge(traj.reset_index(), on="Id", how="left")
n_no_traj = int(u["trajectory"].isna().sum())
if n_no_traj:
    print(f"Note: {n_no_traj} user(s) have no week-{W_EARLY} or week-{W_LATE} data -> no measurable trajectory.")

SEGMENT_OF = {"regular":    "Completes",
              "never":     "Engaged day-only",
              "irregular": "Inconsistent",
              "abandoned": "Decayers"}
u["segment"] = u["sleep_class"].map(SEGMENT_OF)

assert u["segment"].value_counts().sum() == 33, "segmentation must cover all 33 users"

seg_stats = (u.groupby("segment")
               .agg(users=("Id", "nunique"),
                    days_active=("days_active", "mean"),
                    mean_steps=("mean_steps", "mean"),
                    mean_trajectory=("trajectory", "mean"))
               .round(1))
seg_stats["share_pct"] = (seg_stats["users"] / 33 * 100).round(1)
print("=== Segment profile ===")
print(seg_stats.sort_values("mean_steps", ascending=False))

# profile checks quoted in the report
day_only_steps  = u.loc[u["segment"] == "Engaged day-only", "mean_steps"].mean()
completes_steps = u.loc[u["segment"] == "Completes", "mean_steps"].mean()
dec_traj = u.loc[u["segment"] == "Decayers", "trajectory"].mean()
print(f"\nCheck 'day-only nearly as active as completes': {day_only_steps:,.0f} vs {completes_steps:,.0f} steps/day "
      f"({day_only_steps / completes_steps * 100:.0f}%)")
print(f"Check 'decayers have the most negative trajectory': {dec_traj*100:+.1f}%")

## 5. Finding — sleep abandonment marks the only declining users

The three abandoners are compared with every other segment on weekly trajectory.

In [ ]:
# ---------------------------------------------------------------------------
# 5. DECAY ANALYSIS — the only declining pattern in the data
# ---------------------------------------------------------------------------
seg_traj = u.groupby("segment")["trajectory"].mean().sort_values()
print(f"Mean week-{W_EARLY} -> week-{W_LATE} daily-steps change per segment:")
for seg, val in seg_traj.items():
    print(f"  {seg:<17} {val*100:+6.1f} %")

d = seg_traj.get("Decayers", np.nan)
others = seg_traj.drop("Decayers", errors="ignore")
print(f"\nDecayers: {d*100:+.1f}%  |  every other segment: {others.min()*100:+.1f}% to {others.max()*100:+.1f}%")

# weekly mean steps per segment, for the figure and the trough
seg_week = (daily.merge(u[["Id", "segment"]], on="Id")
                 .groupby(["segment", "week"])["TotalSteps"].mean().unstack())
print("\nMean daily steps per segment per week (week 5 is partial - 3 days):")
print(seg_week.round(0))

dec_w1 = seg_week.loc["Decayers", W_EARLY]
trough_week = seg_week.loc["Decayers"].drop(W_EARLY).idxmin()
dec_trough = seg_week.loc["Decayers", trough_week]
print(f"\nDecayers weekly path: W{W_EARLY} {dec_w1:,.0f} -> W{trough_week} {dec_trough:,.0f} "
      f"steps/day ({(dec_trough / dec_w1 - 1) * 100:+.0f}% at trough)")

## 6. Finding — reality vs the promised benchmarks

The 10,000-step benchmark per segment, and the share of recorded nights under 7 hours of sleep.

In [ ]:
# ---------------------------------------------------------------------------
# 6. BENCHMARKS — 10,000 steps / 7 hours of sleep
# ---------------------------------------------------------------------------
d2 = daily.merge(u[["Id", "segment"]], on="Id")
d2["hits_10k"] = d2["TotalSteps"] >= 10000
bench = (d2.groupby("segment")["hits_10k"].mean() * 100).round(1)
print("% of user-days reaching 10,000 steps, per segment:")
for seg, val in bench.sort_values(ascending=False).items():
    print(f"  {seg:<17} {val:5.1f} %")

sleep["short_night"] = sleep["TotalMinutesAsleep"] < 7 * 60
pct_short = sleep["short_night"].mean() * 100
print(f"\nNights under 7 h of sleep: {int(sleep['short_night'].sum())} of {len(sleep)}"
      f" recorded nights = {pct_short:.1f}%")
print("(Counts only the users who chose to wear the device at night.)")

## 7. Finding — the daily rhythm and the two contact windows

Activity peaks at 6 pm and collapses after 8 pm; the lunch plateau (12-14) and the evening wind-down (20-22) sit outside the peak.

In [ ]:
# ---------------------------------------------------------------------------
# 7. DAILY RHYTHM — peak hour and contact windows
# ---------------------------------------------------------------------------
hourly["hour"] = hourly["ActivityHour"].dt.hour
hour_profile = hourly.groupby("hour")["TotalIntensity"].mean()

peak_hour = int(hour_profile.idxmax())
print(f"Peak activity hour: {peak_hour}:00")

evening_peak = hourly[hourly["hour"].isin([17, 18, 19])]["TotalIntensity"].mean()
after_8pm    = hourly[hourly["hour"] >= 20]["TotalIntensity"].mean()
print(f"Mean intensity 17-19h: {evening_peak:.2f}  ->  20h onwards: {after_8pm:.2f} "
      f"({(after_8pm / evening_peak - 1) * 100:+.0f}%)")

for label, lo, hi in [("lunch window 12-14", 12, 14), ("wind-down window 20-22", 20, 22)]:
    in_window  = hour_profile.loc[lo:hi - 1].mean()
    print(f"{label:<24} mean intensity {in_window:.2f} (vs daily mean {hour_profile.mean():.2f})")

## 8. Figures — the four presentation charts

In [ ]:
# ---------------------------------------------------------------------------
# 8. FIGURES
# ---------------------------------------------------------------------------
SEG_ORDER  = ["Completes", "Engaged day-only", "Inconsistent", "Decayers"]
SEG_COLORS = {"Completes": "#2a9d8f", "Engaged day-only": "#e9c46a",
              "Inconsistent": "#8ab17d", "Decayers": "#e76f51"}

# --- Fig 3: sleep adoption (the gateway metric) -----------------------------
fig, ax = plt.subplots(figsize=(7, 4.2))
order  = ["never", "abandoned", "irregular", "regular"]
labels = ["Never worn\nat night", "Tried, abandoned\nwithin days",
          "Irregular\nnight wearers", "Regular night\nwearers"]
vals = [int(counts[k]) for k in order]
pcts = [v / 33 * 100 for v in vals]
bars = ax.bar(labels, pcts,
              color=["#b5838d", "#e76f51", "#f4a261", "#2a9d8f"])
for b, v, p in zip(bars, vals, pcts):
    ax.text(b.get_x() + b.get_width() / 2, p + 0.8,
            f"{v} users\n{p:.0f}%", ha="center", fontsize=9)
ax.set_ylabel("Share of the 33 daily-active users (%)")
ax.set_title("Sleep tracking is the most fragile feature of the tracker", fontsize=12)
ax.set_ylim(0, max(pcts) * 1.32)
fig.savefig("share_fig3_night.png")
plt.show()

# --- Fig 1: four segments, weekly trajectories ------------------------------
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for seg in SEG_ORDER:
    line = seg_week.loc[seg].dropna()
    ax.plot(line.index, line.values, marker="o", label=seg, color=SEG_COLORS[seg])
ax.set_xticks(sorted(seg_week.columns))
ax.set_xticklabels(["W1", "W2", "W3", "W4", "W5\n(partial)"][:len(seg_week.columns)],
                   fontsize=9)
ax.set_xlabel("Study week")
ax.set_ylabel("Mean daily steps")
ax.legend(frameon=False, fontsize=9)
ax.set_title("Four behavioural segments - only the Decayers decline", fontsize=12)
fig.savefig("share_fig1_segments.png")
plt.show()

# --- Fig 4: reality vs the promised benchmarks ------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = axes[0]
bench_ordered = bench.reindex(SEG_ORDER)
ax.bar(range(len(SEG_ORDER)), bench_ordered.values,
       color=[SEG_COLORS[s] for s in SEG_ORDER])
ax.axhline(100, ls="--", color="grey", lw=1)
ax.set_xticks(range(len(SEG_ORDER)))
ax.set_xticklabels([s.replace(" ", "\n") for s in SEG_ORDER], fontsize=8)
for i, v in enumerate(bench_ordered.values):
    ax.text(i, v + 1.5, f"{v:.0f}%", ha="center", fontsize=9)
ax.set_ylabel("% of user-days reaching 10,000 steps")
ax.set_ylim(0, 110)
ax.set_title("The 10,000-step promise", fontsize=11)

ax = axes[1]
ax.hist(sleep["TotalMinutesAsleep"] / 60, bins=20, color="#264653", alpha=0.85)
ax.axvline(7, ls="--", color="#e76f51", lw=1.5)
ax.text(7.1, ax.get_ylim()[1] * 0.9, f"{pct_short:.0f}% of nights\nunder 7 h",
        color="#e76f51", fontsize=9)
ax.axvline(sleep["TotalMinutesAsleep"].mean() / 60, color="#264653", lw=1.5)
ax.set_xlabel("Hours asleep per recorded night")
ax.set_ylabel("Nights")
ax.set_title("The 7-hour sleep promise", fontsize=11)
fig.suptitle("Reality sits far below the benchmarks the industry promises",
             fontsize=12, y=1.03)
fig.savefig("share_fig4_gap.png")
plt.show()

# --- Fig 5: daily rhythm and contact windows --------------------------------
fig, ax = plt.subplots(figsize=(8.5, 4))
ax.plot(hour_profile.index, hour_profile.values, color="#264653", lw=2,
        marker="o", markersize=3)
for lo, hi in [(12, 14), (20, 22)]:
    ax.axvspan(lo, hi, color="#2a9d8f", alpha=0.15)
    ax.text(lo + 0.15, hour_profile.max() * 0.97, f"{lo}:00-{hi}:00",
            color="#1b6b5f", fontsize=9, fontweight="bold")
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hour of day")
ax.set_ylabel("Mean intensity")
ax.set_title("Two natural contact windows: lunch (12-14) and evening wind-down (20-22)",
             fontsize=11)
fig.savefig("share_fig5_windows.png")
plt.show()

print("Figures saved: share_fig3_night.png, share_fig1_segments.png, "
      "share_fig4_gap.png, share_fig5_windows.png")

## 9. Verified numbers — the single source of truth for the written deliverables

This cell prints every number quoted in README.md, REPORT.md, ANALYZE.md, PREPARE.md and CLEANING_LOG.md, with its definition. If any written document disagrees with this cell, the notebook is right.

In [ ]:
# ---------------------------------------------------------------------------
# 9. VERIFIED NUMBERS — source of truth for all written deliverables
# ---------------------------------------------------------------------------
n_never, n_aband, n_irr, n_reg = (int(counts[k])
                                  for k in ["never", "abandoned", "irregular", "regular"])
seg_counts = u["segment"].value_counts()

print("=== VERIFIED NUMBERS (source for README / REPORT / ANALYZE / CLEANING_LOG) ===")
print(f"Users: {n_act} verified (Kaggle announces 30) | sleepDay users: {n_slp} "
      f"| rows after dedup: {len(sleep)} | distinct days: {daily['ActivityDate'].nunique()}")
print(f"Night-wearing classes: never {n_never} ({n_never/33*100:.0f}%) | "
      f"abandoned {n_aband} ({n_aband/33*100:.0f}%) | "
      f"irregular {n_irr} ({n_irr/33*100:.0f}%) | "
      f"regular {n_reg} ({n_reg/33*100:.0f}%)  -> sum {n_never + n_aband + n_irr + n_reg} of 33")
print(f"Sleep tracked at least once: {n_slp} of 33 ({n_slp/33*100:.0f}%); "
      f"durable adoption: {n_reg} of 33 ({n_reg/33*100:.0f}%)")
print("\nSegments:")
for seg in SEG_ORDER:
    steps = u.loc[u["segment"] == seg, "mean_steps"].mean()
    print(f"  {seg:<17} {int(seg_counts[seg])} users ({int(seg_counts[seg])/33*100:>3.0f}%) | "
          f"mean {steps:>7,.0f} steps/day | trajectory {seg_traj.get(seg, np.nan)*100:+6.1f}%")
print(f"\nDecayers weekly path: W{W_EARLY} {dec_w1:,.0f} -> W{trough_week} {dec_trough:,.0f} "
      f"steps/day ({(dec_trough/dec_w1 - 1)*100:+.0f}% at trough); "
      f"week{W_EARLY}->week{W_LATE}: {d*100:+.0f}%")
print("10,000-step benchmark, % of user-days: " +
      " | ".join(f"{s}: {bench[s]:.1f}%" for s in SEG_ORDER))
print(f"Nights under 7 h: {int(sleep['short_night'].sum())} of {len(sleep)} ({pct_short:.1f}%)")
print(f"Peak hour: {peak_hour}:00 | intensity 17-19h: {evening_peak:.1f} | "
      f"after 20h: {after_8pm:.1f} ({(after_8pm/evening_peak - 1)*100:+.0f}%)")

with open("VERIFIED_NUMBERS.txt", "w") as f:
    f.write("Bellabeat case study - verified numbers\n")
    f.write(f"Window: {START.date()} to {END.date()} | users: 33\n")
print("\nSaved: VERIFIED_NUMBERS.txt")

## 10. What this analysis does NOT claim

- It does not describe Bellabeat's actual customers — the sample is 33 US Mechanical Turk workers from 2016, gender undocumented.
- It does not establish causality — the declining-activity pattern rests on 3 users and co-occurs with sleep abandonment; a common cause is not excluded.
- It does not measure sleep quality — accelerometer sleep detects immobility; durations are overestimates by construction.
- It does not validate the 2026 market recommendations — each is a hypothesis to test on Bellabeat's own user data, each with its stated limit.

Analytical chain: ASK.md → PREPARE.md → CLEANING_LOG.md → ANALYZE.md → REPORT.md + SOURCES.md — with this notebook behind every number.